In [16]:
import os
import pandas as pd

BASE_DIR = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data"

station_cols = [
    "CodigoEstacion", "NombreEstacion", "Departamento",
    "Municipio", "ZonaHidrografica", "Latitud", "Longitud"
]

data_2026_2 = pd.read_excel(
    "C:/Users/Usuario/OneDrive - Global Green Growth Institute/Documentos/2025/Outputs/Output4/Indicadores/precipitacion diaria/precipitacion_diaria.xlsx"
)

df_daily_2026_2 = data_2026_2.rename(columns={
    "codigoestacion":   "CodigoEstacion",
    "nombreestacion":   "NombreEstacion",
    "departamento":     "Departamento",
    "municipio":        "Municipio",
    "zonahidrografica": "ZonaHidrografica",
    "latitud":          "Latitud",
    "longitud":         "Longitud",
})

df_daily_2026_2["fecha"] = pd.to_datetime(df_daily_2026_2["fecha"]).dt.normalize()

df_daily_2026_2 = (
    df_daily_2026_2[
        station_cols + ["fecha", "precip_min_10min", "precip_max_10min",
                         "precip_media_10min", "precip_acum_diaria"]
    ]
    .sort_values(["CodigoEstacion", "fecha"])
)

print("df_daily_2026_2 shape:", df_daily_2026_2.shape)
df_daily_2026_2.head()


df_hist_acumulado = pd.read_csv(os.path.join(BASE_DIR, "precipitacion_diaria_acumulada_202603.csv"))


df_daily_vf = pd.concat([df_hist_acumulado, df_daily_2026_2])
df_daily_vf = df_daily_vf.drop_duplicates(["CodigoEstacion", "fecha"])
print("df_daily_vf shape:", df_daily_vf.shape)
df_daily_vf['fecha'] = pd.to_datetime(df_daily_vf['fecha'])
df_daily_vf['mes']   = df_daily_vf['fecha'].dt.month
df_daily_vf.head()


df_daily_2026_2 shape: (69221, 12)
df_daily_vf shape: (1288423, 12)


,CodigoEstacion,NombreEstacion,Departamento,Municipio,ZonaHidrografica,Latitud,Longitud,fecha,precip_min_10min,precip_max_10min,precip_media_10min,precip_acum_diaria,mes
0,11025501,CARMEN DE ATRATO - AUT,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.888719,-76.145167,2022-12-19,0.0,0.1,0.000526,0.1,12
1,11025501,CARMEN DE ATRATO - AUT,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.888719,-76.145167,2022-12-20,0.0,0.0,0.000000,0.0,12
2,11025501,CARMEN DE ATRATO - AUT,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.888719,-76.145167,2022-12-21,0.0,0.1,0.000347,0.1,12
3,11025501,CARMEN DE ATRATO - AUT,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.888719,-76.145167,2022-12-22,0.0,0.1,0.000347,0.1,12
4,11025501,CARMEN DE ATRATO - AUT,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.888719,-76.145167,2022-12-23,0.0,0.0,0.000000,0.0,12


In [22]:
# ─────────────────────────────────────────────────────────────────────
# Rellena DÍAS FALTANTES por estación para UN TRAMO que tú defines.
#
# Flujo manual:
#   1. Pon VENTANA_INI / VENTANA_FIN al tramo que quieras (p. ej. un mes).
#   2. Corre la celda. Descarga solo los días faltantes de ese tramo.
#   3. Guarda UN ARCHIVO PROPIO por tramo:
#        precip_faltantes_<ini>_<fin>.xlsx
#   4. Cambia el tramo y repite. Cada tramo = su archivo, no se pisan.
#
# A prueba de cortes (sin parquet, todo CSV):
#   * Cada 10 estaciones vuelca un checkpoint CSV en disco.
#   * Si se corta, vuelve a correr el MISMO tramo: lee el checkpoint,
#     salta las estaciones ya hechas y continúa.
# ─────────────────────────────────────────────────────────────────────
import os
import time
import requests
import numpy as np
import pandas as pd

# ===== TRAMO QUE CONFIGURAS A MANO =================================
VENTANA_INI = pd.Timestamp("2026-01-01")
VENTANA_FIN = pd.Timestamp("2026-03-06")
# ejemplos de tramos siguientes:
#   VENTANA_INI = pd.Timestamp("2026-07-06"); VENTANA_FIN = pd.Timestamp("2026-08-06")
#   VENTANA_INI = pd.Timestamp("2026-06-06"); VENTANA_FIN = pd.Timestamp("2026-07-06")
# =================================================================

PRECIP_ID  = "s54a-sgyg"       # Precipitación (observaciones IDEAM)
BASE       = "https://www.datos.gov.co/resource"
APP_TOKEN  = os.getenv("SODA_APP_TOKEN", "")    # opcional: sube el rate limit
PAGE       = 50000
FLUSH_CADA = 10                # guarda checkpoint cada N estaciones

OUT_DIR = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data\faltantes_por_tramo"
os.makedirs(OUT_DIR, exist_ok=True)

HEADERS = {"X-App-Token": APP_TOKEN} if APP_TOKEN else {}
VALOR_COLS = ["precip_min_10min", "precip_max_10min", "precip_media_10min", "precip_acum_diaria"]

try:
    station_cols
except NameError:
    station_cols = ["CodigoEstacion", "NombreEstacion", "Departamento",
                    "Municipio", "ZonaHidrografica", "Latitud", "Longitud"]

suf       = f"{VENTANA_INI.date()}_{VENTANA_FIN.date()}"
OUT_XLSX  = os.path.join(OUT_DIR, f"precip_faltantes_{suf}.xlsx")
CKPT      = os.path.join(OUT_DIR, f"_ckpt_{suf}.csv")
CKPT_EST  = os.path.join(OUT_DIR, f"_ckpt_{suf}_estaciones.csv")


def soda_get(dataset_id, params):
    """Paginación completa de un query SODA -> DataFrame."""
    filas, offset = [], 0
    while True:
        p = dict(params)
        p.update({"$limit": PAGE, "$offset": offset})
        r = requests.get(f"{BASE}/{dataset_id}.json", params=p, headers=HEADERS, timeout=120)
        r.raise_for_status()
        chunk = r.json()
        if not chunk:
            break
        filas.extend(chunk)
        offset += PAGE
        if len(chunk) < PAGE:
            break
        time.sleep(0.2)
    return pd.DataFrame(filas)


def a_rangos(dias):
    """Lista de fechas -> tramos [(ini, fin), ...] de días consecutivos."""
    dias = sorted(pd.Timestamp(d).normalize() for d in dias)
    tramos, ini, prev = [], dias[0], dias[0]
    for d in dias[1:]:
        if (d - prev).days == 1:
            prev = d
        else:
            tramos.append((ini, prev))
            ini = prev = d
    tramos.append((ini, prev))
    return tramos


# ---- Días que YA existen por estación (desde df_daily_vf) ----------
base = df_daily_vf.assign(
    CodigoEstacion=lambda d: d["CodigoEstacion"].astype(str).str.strip(),
    fecha=lambda d: pd.to_datetime(d["fecha"]).dt.normalize(),
)
meta_est = base[station_cols].drop_duplicates("CodigoEstacion").set_index("CodigoEstacion")
dias_por_est = base.groupby("CodigoEstacion")["fecha"].agg(lambda s: set(s))
codigos = sorted(dias_por_est.index)

calendario = pd.date_range(VENTANA_INI, VENTANA_FIN, freq="D")

# ---- Reanudar desde checkpoint (si existe) ------------------------
if os.path.exists(CKPT_EST):
    hechas = set(pd.read_csv(CKPT_EST, dtype=str)["CodigoEstacion"])
    ckpt_df = pd.read_csv(CKPT, parse_dates=["fecha"]) if os.path.exists(CKPT) else pd.DataFrame()
    print(f"Reanudando tramo {suf}: {len(hechas)} estaciones ya hechas")
else:
    hechas, ckpt_df = set(), pd.DataFrame()

pendientes = [c for c in codigos if c not in hechas]
print(f"Tramo {suf} ({len(calendario)} días) | pendientes: {len(pendientes)} / {len(codigos)}")

buffer = [ckpt_df] if not ckpt_df.empty else []


def _flush():
    if buffer:
        pd.concat(buffer, ignore_index=True).to_csv(CKPT, index=False)
    pd.DataFrame({"CodigoEstacion": sorted(hechas)}).to_csv(CKPT_EST, index=False)


# ---- Loop -------------------------------------------------------
errores = []
try:
    for i, cod in enumerate(pendientes, 1):
        faltantes = sorted(set(calendario) - dias_por_est.loc[cod])
        if faltantes:
            partes = []
            for d0, d1 in a_rangos(faltantes):
                ini = d0.strftime("%Y-%m-%dT00:00:00.000")
                fin = (d1 + pd.Timedelta(days=1)).strftime("%Y-%m-%dT00:00:00.000")
                try:
                    df = soda_get(PRECIP_ID, {
                        "$select": "codigoestacion,fechaobservacion,valorobservado",
                        "$where": (f"codigoestacion = '{cod}' "
                                   f"AND fechaobservacion >= '{ini}' AND fechaobservacion < '{fin}'"),
                        "$order": "fechaobservacion",
                    })
                except requests.HTTPError as e:
                    errores.append((cod, ini, fin, str(e)))
                    continue
                if not df.empty:
                    partes.append(df)

            if partes:
                obs = pd.concat(partes, ignore_index=True)
                obs["fechaobservacion"] = pd.to_datetime(obs["fechaobservacion"], errors="coerce")
                obs["valorobservado"]   = pd.to_numeric(obs["valorobservado"], errors="coerce")
                obs = obs.dropna(subset=["fechaobservacion"])
                obs["fecha"] = obs["fechaobservacion"].dt.normalize()
                obs = obs[obs["fecha"].isin(faltantes)]
                if not obs.empty:
                    diario = (
                        obs.groupby("fecha", as_index=False)
                           .agg(precip_min_10min=("valorobservado", "min"),
                                precip_max_10min=("valorobservado", "max"),
                                precip_media_10min=("valorobservado", "mean"),
                                precip_acum_diaria=("valorobservado", "sum"))
                    )
                    diario["CodigoEstacion"] = cod
                    for c in station_cols:
                        if c != "CodigoEstacion":
                            diario[c] = meta_est.loc[cod, c] if cod in meta_est.index else np.nan
                    buffer.append(diario[station_cols + ["fecha"] + VALOR_COLS])

        hechas.add(cod)
        if i % FLUSH_CADA == 0:
            _flush()
            n = sum(len(x) for x in buffer)
            print(f"  checkpoint: {i}/{len(pendientes)} | filas: {n}")

    _flush()
    print(f"\nTramo {suf} COMPLETO.")

except (KeyboardInterrupt, Exception) as e:
    _flush()
    print(f"\n⚠ Interrumpido ({type(e).__name__}: {e}). Checkpoint guardado — "
          f"re-corre el MISMO tramo para continuar.")
    raise


# ---- Archivo propio del tramo ---------------------------------
df_tramo = (
    pd.concat(buffer, ignore_index=True)[station_cols + ["fecha"] + VALOR_COLS]
    if buffer else
    pd.DataFrame(columns=station_cols + ["fecha"] + VALOR_COLS)
)
df_tramo["fecha"] = pd.to_datetime(df_tramo["fecha"]).dt.normalize()
df_tramo = (df_tramo.drop_duplicates(["CodigoEstacion", "fecha"])
                    .sort_values(["CodigoEstacion", "fecha"])
                    .reset_index(drop=True))
df_tramo["mes"] = df_tramo["fecha"].dt.month

df_tramo.to_excel(OUT_XLSX, index=False)
print(f"Guardado: {OUT_XLSX}  ({df_tramo.shape[0]} filas, "
      f"{df_tramo['CodigoEstacion'].nunique()} estaciones)")

if errores:
    pd.DataFrame(errores, columns=["CodigoEstacion", "ini", "fin", "error"]).to_csv(
        os.path.join(OUT_DIR, f"errores_{suf}.csv"), index=False)

# Cuando el tramo quede OK, puedes borrar _ckpt_{suf}.csv y _ckpt_{suf}_estaciones.csv


Reanudando tramo 2026-01-01_2026-03-06: 32 estaciones ya hechas
Tramo 2026-01-01_2026-03-06 (65 días) | pendientes: 989 / 1021
  checkpoint: 10/989 | filas: 0

⚠ Interrumpido (ReadTimeout: HTTPSConnectionPool(host='www.datos.gov.co', port=443): Read timed out. (read timeout=120)). Checkpoint guardado — re-corre el MISMO tramo para continuar.


ReadTimeout: HTTPSConnectionPool(host='www.datos.gov.co', port=443): Read timed out. (read timeout=120)